In [1]:
# ── DAY 12 | BASELINE MODELS ──
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [2]:
# ── STEP 2: Load dataset ──
df = pd.read_csv(r'D:\Python project\data\European_Bank_Segmented.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nChurn rate: {df['Exited'].mean()*100:.2f}%")

Shape: (10000, 21)

Columns:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'AgeGroup', 'CreditBand', 'TenureGroup', 'BalanceSegment', 'CLV_Score', 'CLV_Segment', 'EngagementScore', 'Balance_Salary_Ratio', 'Cluster', 'Persona']

Churn rate: 20.37%


In [3]:
# ── STEP 3: Prepare features ──
# Drop target, leakage, redundant columns
drop_cols = ['Exited', 'Cluster', 'Persona',
             'AgeGroup', 'CreditBand', 'TenureGroup',
             'BalanceSegment', 'CLV_Segment']

X = df.drop(columns=drop_cols)
y = df['Exited']

# Encode Gender
X['Gender'] = X['Gender'].map({'Male': 0, 'Female': 1})

# One-Hot Encode Geography
X = pd.get_dummies(X, columns=['Geography'], drop_first=False)

print(f"Features shape: {X.shape}")
print(f"\nFeature columns:\n{list(X.columns)}")
print(f"\nTarget distribution:\n{y.value_counts()}")

Features shape: (10000, 15)

Feature columns:
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'CLV_Score', 'EngagementScore', 'Balance_Salary_Ratio', 'Geography_France', 'Geography_Germany', 'Geography_Spain']

Target distribution:
Exited
0    7963
1    2037
Name: count, dtype: int64


In [4]:
# ── STEP 4: Train/Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"\nTraining churn rate: {y_train.mean()*100:.2f}%")
print(f"Test churn rate:     {y_test.mean()*100:.2f}%")

Training set: (8000, 15)
Test set:     (2000, 15)

Training churn rate: 20.38%
Test churn rate:     20.35%


In [5]:
# ── STEP 5: Apply SMOTE ──
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {X_train.shape[0]} samples")
print(f"After SMOTE:  {X_train_bal.shape[0]} samples")
print(f"\nClass distribution after SMOTE:")
print(pd.Series(y_train_bal).value_counts())
print(f"\nNew churn rate: {y_train_bal.mean()*100:.2f}%")

Before SMOTE: 8000 samples
After SMOTE:  12740 samples

Class distribution after SMOTE:
Exited
1    6370
0    6370
Name: count, dtype: int64

New churn rate: 50.00%


In [6]:
# ── STEP 6: Train Logistic Regression ──
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_bal, y_train_bal)

y_pred_lr = lr_model.predict(X_test)

print("LOGISTIC REGRESSION RESULTS")
print("="*45)
print(classification_report(y_test, y_pred_lr))

LOGISTIC REGRESSION RESULTS
              precision    recall  f1-score   support

           0       0.84      0.84      0.84      1593
           1       0.39      0.40      0.39       407

    accuracy                           0.75      2000
   macro avg       0.62      0.62      0.62      2000
weighted avg       0.75      0.75      0.75      2000

